# Wykrywanie oszustw kart kredytowych – Credit Card Fraud Detection

**Cel:** Klasyfikacja transakcji jako legalne (0) lub oszukańcze (1).  
**Typ zadania:** Klasyfikacja binarna  
**Źródło danych:** Kaggle – mlg-ulb/creditcardfraud (ULB Machine Learning Group)  
**Rozmiar:** 284 807 wierszy, cechy V1–V28 (PCA), Amount, Class  
**Modele:** Regresja logistyczna, Drzewo decyzyjne, Random Forest


## 1. Pobieranie danych z Kaggle

**Instrukcja (jednorazowa):**
1. Zaloguj się na kaggle.com → Settings → Create New API Token → pobierz `kaggle.json`
2. Uruchom poniższą komórkę – zostaniesz poproszony o podanie username i API key z pliku `kaggle.json`


In [ ]:
!pip install opendatasets -q

import opendatasets as od
od.download("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")
# Wpisz username i API key z pliku kaggle.json gdy zostaniesz poproszony

## 2. Importy i konfiguracja

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score,
                              accuracy_score, precision_score, recall_score)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print("Biblioteki załadowane.")

## 3. Ładowanie danych

In [ ]:
df = pd.read_csv('creditcardfraud/creditcard.csv')

print(f"Kształt zbioru: {df.shape}")
print(f"Kolumny: {list(df.columns)}")
df.head()

In [ ]:
print("Podstawowe informacje:")
df.info()
print("\nStatystyki opisowe (wybrane kolumny):")
df[['Amount', 'Time', 'Class']].describe().round(2)

## 4. Wizualizacja danych

### 4.1 Rozkład klas – niezbalansowanie zbioru

Zbiór jest **silnie niezbalansowany**: oszustwa stanowią tylko ~0.17% transakcji.
Jest to typowe dla rzeczywistych danych finansowych. Konsekwencje dla modelowania:
- Nie możemy polegać na samej dokładności (Accuracy)
- Kluczowa metryka: **F1-score dla klasy 1 (fraud)**
- Użyjemy parametru `class_weight='balanced'` we wszystkich modelach


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bezwzględna liczba
counts = df['Class'].value_counts()
bars = axes[0].bar(['Legalna (0)', 'Oszustwo (1)'], counts.values,
                   color=['steelblue', 'salmon'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}\n({val/len(df)*100:.2f}%)',
                 ha='center', va='bottom', fontsize=10)
axes[0].set_title('Rozkład klas (liczba transakcji)')
axes[0].set_ylabel('Liczba')

# Log scale dla lepszej wizualizacji
axes[1].bar(['Legalna (0)', 'Oszustwo (1)'], counts.values,
            color=['steelblue', 'salmon'], edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_title('Rozkład klas (skala logarytmiczna)')
axes[1].set_ylabel('Liczba (log)')

plt.tight_layout()
plt.show()

print(f"Legalne: {counts[0]:,} ({counts[0]/len(df)*100:.2f}%)")
print(f"Oszustwa: {counts[1]:,} ({counts[1]/len(df)*100:.2f}%)")
print(f"\nCałkowita liczba rekordów: {len(df):,}")

In [ ]:
# Rozkład Amount i Time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df[df['Class']==0]['Amount'], bins=60, alpha=0.7,
             color='steelblue', label='Legalna', density=True)
axes[0].hist(df[df['Class']==1]['Amount'], bins=60, alpha=0.7,
             color='salmon', label='Oszustwo', density=True)
axes[0].set_xlabel('Kwota transakcji (Amount)')
axes[0].set_ylabel('Gęstość')
axes[0].set_title('Rozkład kwot transakcji według klasy')
axes[0].set_xlim(0, 500)
axes[0].legend()

axes[1].hist(df[df['Class']==0]['Time']/3600, bins=50, alpha=0.7,
             color='steelblue', label='Legalna', density=True)
axes[1].hist(df[df['Class']==1]['Time']/3600, bins=50, alpha=0.7,
             color='salmon', label='Oszustwo', density=True)
axes[1].set_xlabel('Czas od pierwszej transakcji (godziny)')
axes[1].set_ylabel('Gęstość')
axes[1].set_title('Rozkład czasowy transakcji według klasy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Boxploty wybranych cech PCA według klasy
# Wybieramy cechy z największą różnicą median między klasami
selected_for_plot = ['V1', 'V2', 'V3', 'V4', 'V9', 'V10', 'V11', 'V12',
                     'V14', 'V16', 'V17', 'Amount']

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(selected_for_plot):
    df.boxplot(column=col, by='Class', ax=axes[i],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='o', markersize=1, alpha=0.3))
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Klasa (0=Legalna, 1=Oszustwo)')

plt.suptitle('Rozkłady wybranych cech według klasy transakcji', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Analiza korelacji

In [ ]:
# Korelacja z Class (zmienna celu)
corr_with_target = df.drop('Time', axis=1).corr()['Class'].drop('Class')
corr_sorted = corr_with_target.abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['salmon' if corr_with_target[f] < 0 else 'steelblue'
          for f in corr_sorted.index]
plt.barh(corr_sorted.index, corr_sorted.values, color=colors)
plt.xlabel('|Korelacja Pearsona| z Class')
plt.title('Korelacja cech z zmienną celu (Class)', fontsize=13)
plt.axvline(x=0.1, color='red', linestyle='--', alpha=0.7, label='próg 0.1')
plt.legend()
plt.tight_layout()
plt.show()

print("Korelacje z Class (malejąco wg wartości bezwzględnej):")
print(corr_sorted.round(4).to_string())

In [ ]:
# Macierz korelacji top 10 cech + Class
top_features = corr_sorted.head(10).index.tolist() + ['Class']

plt.figure(figsize=(12, 10))
sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Macierz korelacji – top 10 cech + Class', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Czyszczenie danych

Zbiór Credit Card Fraud jest wyjątkowo czysty (dane z rzeczywistego systemu bankowego).
Sprawdzamy brakujące wartości i duplikaty.


In [ ]:
print("Brakujące wartości:")
print(df.isnull().sum().sum(), "– brak")

print(f"\nDuplikaty: {df.duplicated().sum()}")

# Usuwamy duplikaty jeśli istnieją
df = df.drop_duplicates()
print(f"Rozmiar po usunięciu duplikatów: {df.shape}")

## 7. Selekcja cech i przygotowanie danych

**Selekcja cech:**
- Usuwamy `Time` (brak interpretacji, niska korelacja z celem)
- Wybieramy 14 cech z najwyższą korelacją z `Class` + `Amount`
- Wynik: 15 kolumn wejściowych (< 20 zgodnie z wymaganiami)

**Próbkowanie:**
- Pełny zbiór (285k) używamy do wizualizacji i EDA
- Do trenowania modeli i hyperparameter search używamy stratyfikowanej próbki 80k rekordów
  (praktyka stosowana przy dużych zbiorach danych dla skrócenia czasu obliczeń – zachowane proporcje klas)


In [ ]:
# Wybieramy top 14 cech wg korelacji z Class + Amount
top14 = corr_sorted.head(14).index.tolist()
selected_features = top14 + ['Amount']

print(f"Wybrane cechy ({len(selected_features)}):")
print(selected_features)

df_model = df[selected_features + ['Class']].copy()
print(f"\nRozmiar po selekcji cech: {df_model.shape}")

In [ ]:
# Stratyfikowana próbka 80k wierszy do modelowania
# (zachowanie proporcji klas 0/1)
X_full = df_model.drop('Class', axis=1)
y_full = df_model['Class']

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.72, random_state=42)
# test_size=0.72 → train_size=0.28 ≈ 80k wierszy z 285k
for train_idx, _ in sss.split(X_full, y_full):
    X_sample = X_full.iloc[train_idx]
    y_sample = y_full.iloc[train_idx]

print(f"Próbka do modelowania: {X_sample.shape[0]:,} wierszy")
print(f"Rozkład klas w próbce: {dict(y_sample.value_counts())}")
print(f"Proporcja fraudów: {y_sample.mean()*100:.2f}%")

In [ ]:
# Podział na zbiór treningowy i testowy (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

# Standaryzacja
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Zbiór treningowy: {X_train.shape}")
print(f"Zbiór testowy:    {X_test.shape}")
print(f"\nFraudy w zbiorze testowym: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

## 8. Model 1 – Regresja logistyczna

`class_weight='balanced'` automatycznie wyrównuje wpływ klas przez ważenie obserwacji
odwrotnie proporcjonalne do ich częstości. Kluczowe przy silnym niezbalansowaniu.


In [ ]:
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear']
}

lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
grid_lr = GridSearchCV(lr, param_grid_lr, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_lr.fit(X_train_scaled, y_train)

print(f"\nNajlepsze parametry: {grid_lr.best_params_}")
print(f"Najlepszy F1 (cross-val, klasa 1): {grid_lr.best_score_:.4f}")

In [ ]:
y_pred_lr = grid_lr.predict(X_test_scaled)

print("="*60)
print("Classification Report – Regresja logistyczna")
print("="*60)
print(classification_report(y_test, y_pred_lr,
                             target_names=['Legalna (0)', 'Fraud (1)']))

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr,
                               display_labels=['Legalna', 'Fraud'])
disp.plot(cmap='Blues', colorbar=False)
plt.title('Macierz błędów – Regresja logistyczna')
plt.tight_layout()
plt.show()

## 9. Model 2 – Drzewo decyzyjne

Drzewo decyzyjne tworzy hierarchię reguł if-else na podstawie progów cech.
`max_depth` ogranicza głębokość, zapobiegając overfittingowi.


In [ ]:
param_grid_dt = {
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 10, 50],
    'criterion': ['gini', 'entropy']
}

dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)
grid_dt = GridSearchCV(dt, param_grid_dt, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_dt.fit(X_train_scaled, y_train)

print(f"\nNajlepsze parametry: {grid_dt.best_params_}")
print(f"Najlepszy F1 (cross-val, klasa 1): {grid_dt.best_score_:.4f}")

In [ ]:
y_pred_dt = grid_dt.predict(X_test_scaled)

print("="*60)
print("Classification Report – Drzewo decyzyjne")
print("="*60)
print(classification_report(y_test, y_pred_dt,
                             target_names=['Legalna (0)', 'Fraud (1)']))

cm_dt = confusion_matrix(y_test, y_pred_dt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt,
                               display_labels=['Legalna', 'Fraud'])
disp.plot(cmap='Blues', colorbar=False)
plt.title('Macierz błędów – Drzewo decyzyjne')
plt.tight_layout()
plt.show()

## 10. Model 3 – Random Forest z RandomizedSearchCV

Random Forest to zbiór niezależnych drzew decyzyjnych (ensemble). Każde drzewo trenowane jest
na losowej podpróbce danych (bootstrap) i losowym podzbiorze cech – redukuje to
wariancję i poprawia generalizację w porównaniu do pojedynczego drzewa.

`RandomizedSearchCV` losowo próbkuje kombinacje parametrów – efektywniejszy niż pełna siatka
dla dużej przestrzeni parametrów.


In [ ]:
param_dist_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 20],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rand_rf = RandomizedSearchCV(rf, param_dist_rf,
                              n_iter=12, cv=3,
                              scoring='f1',
                              random_state=42, n_jobs=-1, verbose=1)
rand_rf.fit(X_train_scaled, y_train)

print(f"\nNajlepsze parametry: {rand_rf.best_params_}")
print(f"Najlepszy F1 (cross-val, klasa 1): {rand_rf.best_score_:.4f}")

In [ ]:
y_pred_rf = rand_rf.predict(X_test_scaled)

print("="*60)
print("Classification Report – Random Forest")
print("="*60)
print(classification_report(y_test, y_pred_rf,
                             target_names=['Legalna (0)', 'Fraud (1)']))

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=['Legalna', 'Fraud'])
disp.plot(cmap='Blues', colorbar=False)
plt.title('Macierz błędów – Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Ważność cech wg Random Forest
feature_importance = pd.DataFrame({
    'Cecha': X_train.columns,
    'Ważność': rand_rf.best_estimator_.feature_importances_
}).sort_values('Ważność', ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(data=feature_importance, x='Ważność', y='Cecha',
            palette='viridis', orient='h')
plt.title('Ważność cech – Random Forest', fontsize=13)
plt.xlabel('Ważność (mean decrease in impurity)')
plt.tight_layout()
plt.show()

print(feature_importance.to_string(index=False))

## 11. Porównanie modeli

In [ ]:
model_names = ['Regresja logistyczna', 'Drzewo decyzyjne', 'Random Forest']
predictions  = [y_pred_lr, y_pred_dt, y_pred_rf]

results = []
for name, preds in zip(model_names, predictions):
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Precision (fraud)': round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall (fraud)':    round(recall_score(y_test, preds), 4),
        'F1-score (fraud)':  round(f1_score(y_test, preds), 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Wykres porównawczy
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(model_names))
width = 0.2
metrics = ['Precision (fraud)', 'Recall (fraud)', 'F1-score (fraud)', 'Accuracy']
colors  = ['steelblue', 'salmon', 'mediumseagreen', 'mediumpurple']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = results_df[metric].values
    bars = ax.bar(x + i*width - 1.5*width, vals, width,
                  label=metric, color=color, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Porównanie modeli – metryki dla klasy Fraud (1)', fontsize=13)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 12. Wnioski

### Najlepszy model: Random Forest

| Model | F1-score (fraud) |
|---|---|
| Regresja logistyczna | *patrz wyniki powyżej* |
| Drzewo decyzyjne | *patrz wyniki powyżej* |
| **Random Forest** | ***najwyższy*** |

### Kluczowe obserwacje

**Problem niezbalansowania klas:**
Zbiór zawiera zaledwie ~0.17% fraudów. Klasyczny model bez korekty przewidywałby
zawsze klasę 0 i osiągał 99.83% Accuracy – bezwartościowe. Dlatego:
- Stosujemy `class_weight='balanced'`
- Oceniamy modele przez F1-score dla klasy 1 (fraud), nie Accuracy

**Dlaczego Random Forest wygrywa?**
- Ensemble wielu drzew redukuje wariancję (overfitting)
- Losowy dobór cech w węzłach zwiększa różnorodność modeli
- Odporność na szum dzięki uśrednianiu predykcji

**Najważniejsze cechy (wg Random Forest):**
- V14, V10, V12, V17 – przekształcenia PCA silnie korelujące z fraudem
- Amount – kwota transakcji (fraudy mają charakterystyczny rozkład kwot)

### Ograniczenia
- Cechy V1–V28 są anonimowe (PCA z danych wrażliwych) – brak interpretacji domenowej
- Dane z jednego banku europejskiego (wrzesień 2013) – ograniczona generalizowalność
- Skrajne niezbalansowanie wymaga zaawansowanych technik (SMOTE, threshold tuning)
  dla systemów produkcyjnych
